# D1.6 · Distinguishing agent from human

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *Security of AI*

Builds on **[D1.5 · Agent telemetry as a data source](https://spbreed.github.io/cyber-commons/lessons/D1.5.html)**.

| | |
|---|---|
| Tools used | OpenSearch, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Build the classifier on timing, sequencing and volume features.

**Why a security engineer needs it.** Your earliest Shadow Autonomy signal is invisible. The control it builds is: behavioural signatures separating agent from inherited human.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The agent holds a human's authority and acts under a human's name. Conventional UEBA reads that as the human behaving strangely, and the entire attribution question — was this a person or their agent — has no field to answer it.

> **At CyberTravels.** CyberTravels acts under Alex's authority and in Alex's name. Conventional UEBA reads that as Alex behaving strangely at 3am. R11.

## 2 · The framework

```
   the log says                    the truth is
   +--------------------+          +---------------------------+
   | user: dana@corp    |          | dana's agent, acting for  |
   | action: deploy     |          | dana, at 03:14            |
   +--------------------+          +---------------------------+

   UEBA reads this as dana behaving strangely.
   the missing field is not "suspicious" - it is "actor_type"
```

Distinguishing agent from human in telemetry matters because the ones you most
need to find are the ones not in any registry (A3.7).

Three signals, none sufficient alone:

- **Regularity** — the coefficient of variation of inter-arrival times. Humans
  are irregular; loops are metronomic.
- **Rate** — sustained multi-action-per-second activity is not typing.
- **Continuity** — software has no evenings.

The honest part of this lesson is the error analysis, because the two error
directions are not symmetric:

- A **human misclassified as an agent** triggers an investigation. Mild cost,
  self-correcting.
- An **agent misclassified as human** stays invisible, which is the entire risk
  you were trying to address.

That asymmetry decides the threshold, and it argues for a lower one than
accuracy-maximisation would give you.

## 3 · The procedure, as a skill

The skill scores five actors on behaviour rather than on what they claim to be, sweeps the threshold, and then picks it by expected cost — because a flagged human costs half an analyst-hour and a missed agent costs forty.

In [ ]:
# skills/detection/agent-versus-human-scoring/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-versus-human-scoring
description: >-
  Score actors on behavioural signals to separate agents from people, sweep the
  threshold, and pick it by expected cost rather than by accuracy. Use when
  deciding whether a session is automated, or when unregistered automation needs
  finding.
allowed-tools: Read, Grep, Glob
---

# Pick the threshold by what each mistake costs

Separating agent from human is a scoring problem with two asymmetric errors: a
flagged human costs an analyst half an hour, and a missed agent costs whatever
an unmonitored automation does. Choosing the threshold by accuracy weights those
equally, which is the one thing you know is wrong.

## When to use this

Finding unregistered automation, deciding whether a session is a person, and
before any control that treats agents differently from users.

## Procedure

**1 — Score on behaviour, not on the user agent string.** Inter-action variance,
rate, breadth, and the share of actions with no preceding read. Anything
self-declared is a claim.

**2 — Score a spread of real actors.** A service indexer, an unknown token, a
person, a person driving an IDE assistant, and an agent deliberately jittered to
look human. The last two are the interesting middle.

**3 — Sweep the threshold and record both errors.** Humans flagged and agents
missed, at each setting. They move in opposite directions and the crossing point
is not the answer.

**4 — Attach a cost to each error and minimise the total.** Analyst hours for a
false positive, expected hours of an unmonitored agent for a false negative. The
chosen threshold now has a justification somebody can argue with.

**5 — Join to the registry.** An actor scoring as an agent and absent from the
registry is the finding worth routing; a registered agent scoring as an agent is
working correctly.

## Output contract

```json
{
  "actors": [{"name": "str", "score": 0.0, "truth": "agent|human|unknown"}],
  "sweep": [{"threshold": 0.0, "humans_flagged": 0, "agents_missed": 0, "expected_cost": 0.0}],
  "costs": {"false_positive_hours": 0.0, "false_negative_hours": 0.0},
  "chosen": {"threshold": 0.0, "why": "str"},
  "registry": {"scored_agent_unregistered": ["str"]}
}
```

## Failure modes

- **Scoring the user agent string.** It is self-declared.
- **Optimising accuracy.** It assumes the two errors cost the same.
- **Flagging registered agents.** They are supposed to look like agents.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/detection/agent-versus-human-scoring/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/detection/agent-versus-human-scoring/scripts/agent_versus_human_scoring.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Score actors on behavioural signals, sweep the threshold, and choose it by expected cost rather than by accuracy.

This is the executable half of the `agent-versus-human-scoring` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import statistics, time
from dataclasses import dataclass

@dataclass
class Event:
    ts: float; actor: str; action: str = "read"

def agent_score(events, actor):
    ev = sorted((e for e in events if e.actor == actor), key=lambda e: e.ts)
    if len(ev) < 3:
        return {"actor": actor, "score": 0.0, "verdict": "insufficient data"}
    gaps = [b.ts - a.ts for a, b in zip(ev, ev[1:])]
    mean = statistics.fmean(gaps)
    cv = (statistics.pstdev(gaps) / mean) if mean else 0.0
    regularity  = max(0.0, 1.0 - min(cv, 1.0))
    rate        = len(ev) / max(ev[-1].ts - ev[0].ts, 1e-9)
    rate_signal = min(rate / 5.0, 1.0)
    span_hours  = (ev[-1].ts - ev[0].ts) / 3600
    continuity  = min(span_hours / 8.0, 1.0)
    score = round(0.5*regularity + 0.3*rate_signal + 0.2*continuity, 3)
    return {"actor": actor, "score": score, "cv": round(cv, 2),
            "rate_per_s": round(rate, 2), "span_h": round(span_hours, 2)}

now = time.time()
POP = {
 "svc-indexer":        ([Event(now + i*0.05, "svc-indexer") for i in range(500)], "agent"),
 "dana@corp":          ([Event(now + t, "dana@corp") for t in
                         (0, 5, 13, 14, 60, 140, 320, 900, 1800, 4000)], "human"),
 "unknown-token-7f3c": ([Event(now + i*1.0, "unknown-token-7f3c") for i in range(400)], "agent"),
 "sam@corp-ide":       ([Event(now + i*2.0, "sam@corp-ide") for i in range(180)], "human"),
 "polite-agent":       ([Event(now + t, "polite-agent") for t in
                         (0, 7, 19, 44, 90, 210, 480, 900, 1700, 3000)], "agent"),
}
print(f"{'actor':22s}{'score':>7}{'cv':>7}{'rate/s':>9}{'span_h':>9}  truth")
print("-" * 62)
for actor, (ev, truth) in POP.items():
    r = agent_score(ev, actor)
    print(f"{actor:22s}{r['score']:>7.3f}{r.get('cv',0):>7}{r.get('rate_per_s',0):>9}"
          f"{r.get('span_h',0):>9}  {truth}")

def evaluate(threshold):
    fp = fn = 0
    for actor, (ev, truth) in POP.items():
        s = agent_score(ev, actor)["score"]
        pred = "agent" if s >= threshold else "human"
        if pred == "agent" and truth == "human": fp += 1
        if pred == "human" and truth == "agent": fn += 1
    return fp, fn

print(f"{'threshold':>10}{'humans flagged':>16}{'agents MISSED':>16}")
print("-" * 44)
for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    fp, fn = evaluate(t)
    flag = "   ← invisible agents" if fn else ""
    print(f"{t:>10.1f}{fp:>16}{fn:>16}{flag}")

print("\nThe two errors cost differently:")
print("   human flagged as agent  → one investigation, ~30 min, self-correcting")
print("   agent flagged as human  → it stays out of your NHI inventory entirely")

COST_FP = 0.5      # analyst-hours per false investigation
COST_FN = 40.0     # expected hours if an unmanaged agent is missed

def expected_cost(threshold):
    fp, fn = evaluate(threshold)
    return fp * COST_FP + fn * COST_FN, fp, fn

print(f"{'threshold':>10}{'FP':>5}{'FN':>5}{'expected cost (hrs)':>22}")
print("-" * 44)
best = None
for t in [x/20 for x in range(4, 19)]:
    c, fp, fn = expected_cost(t)
    if best is None or c < best[1]: best = (t, c)
    if abs(t*20 - round(t*20)) < 1e-9 and (t*10) % 1 == 0:
        print(f"{t:>10.2f}{fp:>5}{fn:>5}{c:>22.1f}")
print(f"\ncost-minimising threshold: {best[0]:.2f} (expected {best[1]:.1f} hrs)")
print("Accuracy-maximising would sit higher and let the polite agent through.")

fp, fn = evaluate(best[0])
print(f"at that threshold: {fp} humans investigated, {fn} agents missed")

# Verify: join against the registry — the score alone is not the finding.
REGISTERED = {"svc-indexer", "dana@corp", "sam@corp-ide"}
threshold = best[0]
findings = []
for actor, (ev, truth) in POP.items():
    s = agent_score(ev, actor)["score"]
    if s >= threshold and actor not in REGISTERED:
        findings.append((actor, s))
print("shadow agents (behaves like software, not in the inventory):")
for a, s in findings:
    print(f"   {a:22s} score={s:.3f}")
assert findings

## What you just proved

The service indexer and unknown token score highest, the human lowest, with the IDE user and the politely-jittered agent in between. The threshold sweep shows humans flagged rising and agents missed falling as the threshold drops. Cost-weighting selects a low threshold, and joining against the registry identifies the unregistered actors as shadow agents.

## Your turn

Set COST_FN honestly for your organisation — it is the expected cost of an unmanaged agent operating undetected for a quarter. That number, not model accuracy, is what should set your threshold.

---

**Next → [D1.7 · Drift monitoring](https://spbreed.github.io/cyber-commons/lessons/D1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*